In [ ]:
def main(datasources, start_date, end_date):
    """
    使用 pandas 计算日内订单簿压力因子。

    返回包含 ['date', 'instrument', 'factor'] 三列的 DataFrame。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # SQL 仅用于读取原始行情字段及 instrument 映射，因子计算均由 pandas 完成。
    volume_columns = [
        *[f"bid_volume{i}" for i in range(1, 6)],
        *[f"ask_volume{i}" for i in range(1, 6)],
    ]
    raw_sql = f"""
    SELECT
        b.date,
        b.instrument_id,
        b.bid_volume1, b.bid_volume2, b.bid_volume3, b.bid_volume4, b.bid_volume5,
        b.ask_volume1, b.ask_volume2, b.ask_volume3, b.ask_volume4, b.ask_volume5,
        i.instrument
    FROM {bar1m} b
    LEFT JOIN all_instruments i USING (instrument_id)
    """
    raw = dai.query(
        raw_sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    if raw.empty:
        return pd.DataFrame({
            "date": pd.Series(dtype="datetime64[ns]"),
            "instrument": pd.Series(dtype="object"),
            "factor": pd.Series(dtype="float64"),
        })

    raw["date"] = pd.to_datetime(raw["date"])
    raw[volume_columns] = raw[volume_columns].fillna(0)

    # 与 SQL 中 EXP(-0.3 * 档位偏移) 完全一致。
    weights = np.exp(-0.3 * np.arange(5))
    bid_columns = [f"bid_volume{i}" for i in range(1, 6)]
    ask_columns = [f"ask_volume{i}" for i in range(1, 6)]
    weight_bid = raw[bid_columns].to_numpy(dtype="float64") @ weights
    weight_ask = raw[ask_columns].to_numpy(dtype="float64") @ weights
    raw["weighted_imbalance"] = (
        (weight_bid - weight_ask) / (weight_bid + weight_ask + 1e-8)
    )

    # 按原 SQL 的左开右闭区间划分固定时间截面。
    hhmmss = (
        raw["date"].dt.hour * 10000
        + raw["date"].dt.minute * 100
        + raw["date"].dt.second
    )
    section_ends = [93000, 100000, 103000, 110000, 113000, 133000, 140000, 143000]
    section_starts = [90000, 93000, 100000, 103000, 110000, 130000, 133000, 140000]
    conditions = [
        (hhmmss > section_start) & (hhmmss <= section_end)
        for section_start, section_end in zip(section_starts, section_ends)
    ]
    raw["time_segment"] = np.select(conditions, section_ends, default=-1)
    raw = raw.loc[raw["time_segment"] != -1].copy()

    raw["trading_day"] = raw["date"].dt.normalize()
    factor = (
        raw.groupby(
            ["trading_day", "time_segment", "instrument_id", "instrument"],
            observed=True,
            as_index=False,
        )["weighted_imbalance"]
        .mean()
        .rename(columns={"weighted_imbalance": "factor"})
    )

    # 将交易日与分段结束时刻组合成固定截面的 date。
    segment_text = factor["time_segment"].astype(str).str.zfill(6)
    factor["date"] = factor["trading_day"] + pd.to_timedelta(
        segment_text.str[:2].astype(int), unit="h"
    ) + pd.to_timedelta(
        segment_text.str[2:4].astype(int), unit="m"
    ) + pd.to_timedelta(
        segment_text.str[4:6].astype(int), unit="s"
    )
    df = factor[["date", "instrument", "factor"]]
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])

    # 与原实现一致：仅保留对应时间截面属于中证 1000 股票池的标的。
    stk_pool = dai.query(
        "SELECT date, instrument FROM cpt_jyc_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"])
    result = pd.merge(df, stk_pool, how="inner", on=["date", "instrument"])

    return result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)


if __name__ == "__main__":
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()
    datasources = {"bar1m": "cpt_jyc_2026_stock_bar1m"}
    start_date = "2020-01-01 00:00:00"
    end_date = "2020-03-01 23:59:59"

    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)
    result = M.jyc_eval._latest(factor_data=factor_data, show=True)
